![](eda.png)

this notebook for understanding and write ***EDA*** for `Car Price` Data set  
this Data set contain columns:    

`ID`,
`Price`,
`Levy`,
`Manufacturer`,
`Model`,
`Prod. year`,
`Category`,
`Leather interior`,
`Fuel type`,
`Engine volume`,
`Mileage`,
`Cylinders`,
`Gear box type`,
`Drive wheels`,
`Doors`,
`Wheel`,
`Color`,
`Airbags`,
`Random_notes`  


## Import Libraries  

first let's import the libraires we will use in this notebook

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
%matplotlib inline 

## Understand the Dataset  

after we import useful libraries, we will try to understand our data set  

- first: Load the dataset into pandas.  

- secend:  
1- ***Missing values***  
2- ***Wrong data types***  
3- ***Duplicates***  
4- ***Outliers***  
5- ***Useless columns***  

In [ ]:
car_sales = pd.read_csv('data/car_price_Dataset.csv')
car_sales.head(10)

In [ ]:

car_sales.info()

From data we can see that   
`Price`, `Lavy`, `Pred-year`, `Mileage`, `Cylinders` are integer data type   
but when print info thay are object or float,  
so we must solve this problem and change their data tyep   

also `Engine volume` column is float and show object in info, so also must change it 

In [ ]:
car_sales['Price'] = car_sales['Price'].str.replace('$','',regex=False)
car_sales['Price'] = car_sales['Price'].astype(int)
print(car_sales['Price'].dtype)

In [ ]:
car_sales['Levy'] = car_sales['Levy'].str.replace('-','0',regex=False).astype(float).astype('Int64') 
print(car_sales['Levy'].dtype)

In [ ]:
car_sales['Prod. year'] = car_sales['Prod. year'].str.replace('unknown','0',regex=False).astype('Int64')
print(car_sales['Prod. year'].dtype)

In [ ]:
car_sales['Mileage'] = car_sales['Mileage'].str.replace('KM','',regex=False).astype(float).astype('Int64')
print(car_sales['Mileage'].dtype)

In [ ]:
car_sales['Cylinders'] = car_sales['Cylinders'].astype('Int64')
print(car_sales['Cylinders'].dtype)


In [ ]:
car_sales['Engine volume'] = car_sales['Engine volume'].str.replace(' Turbo','',regex=False).astype(float)
print(car_sales['Engine volume'].dtype)

In [ ]:
car_sales.info()

In [ ]:
car_sales.describe().T

In [ ]:
car_sales.describe(include=[object]).T

In [ ]:
car_sales.isna().sum()

all the data is clean ***except***  

Levy, Mileage, Color, Random_notes


In [ ]:
car_sales.shape

In [ ]:
100 - ((2424/car_sales.shape[0])*100)

in Levy & Mileage and Color we lose about 87 from the column and   
in Random Notes we loase all the date (might be unuseful feature), so we must drop it  

we can hanle the Levy, Mileage, Color features or drop them

In [ ]:
missing_data = ['Random_notes']
car_sales.drop(columns=missing_data, inplace=True)
car_sales['Mileage'] = car_sales['Mileage'].fillna(int(car_sales['Mileage'].mean()))
car_sales['Levy'] = car_sales['Levy'].fillna(int(car_sales['Levy'].mean()))
car_sales['Color'] = car_sales['Color'].fillna(car_sales['Color'].mode()[0])
print(car_sales.shape)

the ID column is usless so we will drop it

In [ ]:
car_sales.drop(columns=['ID'],axis=1, inplace=True)

In [ ]:
car_sales[car_sales.duplicated()].shape

there are about 682 rows are dublicated

In [ ]:
car_sales.drop_duplicates(keep='first', inplace=True)
car_sales.shape

Explore&Hanle outlier

In [ ]:
numeric_cols = car_sales.select_dtypes(include=['number']).columns

fig, axes = plt.subplots(2,4, figsize=(14,12))

axes = axes.flatten()

for i, col in enumerate(numeric_cols):
    axes[i].boxplot(car_sales[col])
    axes[i].set_title(col)


plt.suptitle("Box Plot for Numeric Features")
plt.tight_layout()
plt.show()


In [ ]:
numeric_cols = car_sales.select_dtypes(include=['number']).columns

for col in numeric_cols:
    q1 = car_sales[col].quantile(0.25)
    q3 = car_sales[col].quantile(0.75)
    iqr = q3-q1
    lower = q1 - 1.5*iqr
    upper = q3 + 1.5*iqr
    mask = (car_sales[col] < lower) | (car_sales[col] > upper)
        
    if mask.sum() > 0:
        print(f"Column {col} has {mask.sum()} outliers")

as shown the Price, Pred. year, Enfine Volume, Cylinders has outliers  
- `Price` is the target so we shouldn't remove his values so it maybe real values   
- `Pred. year` it maybe there are some cars produced in very oldest/newest years we can imuputate it or change with normal value  
- `Engine Volume` that's maybe because the Engine volume for some cars like Racing cars etc has very powerful engine volume (or another reason)  
- `Cylinders` the data show that there is no big change in values, but maybe as the distinct valure are few values so it show outlier  

## Data Cleaning  
* Handle Missing Values
* Remove Duplicates
* Fix Data Types
* Handle Outliers
* Drop Useless Columns

we already handled missing values  
           remove dublicates  
           fix data types
           drop useless columns
let's handle the outlier

- let's change the outlier values in engine volume to be in range lower&upper  
- remove 0 from year-prod

In [ ]:
q1 = car_sales['Engine volume'].quantile(0.25)
q3 = car_sales['Engine volume'].quantile(0.75)
iqr = q3-q1
lower = q1-1.5*iqr
upper = q3+1.5*iqr

car_sales['Engine volume'] = np.where(car_sales['Engine volume'] < lower, lower, 
                                np.where(car_sales['Engine volume'] > upper, upper, car_sales['Engine volume']))


In [ ]:
zero_year_prod = car_sales[car_sales['Prod. year']==0].index
car_sales.drop(zero_year_prod, axis=0, inplace=True)
car_sales.shape

## Exploratory Data Analysis (EDA)  

- Distribution of prices
- Average price by manufacturer
- Year vs price (scatter plot)
- Mileage bins vs price (box plot).
- Countplot of car colors.

let's visualize our target feature `Price` and see his distribution

In [ ]:
plt.figure(figsize=(10,6))

plt.plot(car_sales['Price'])
plt.xlabel('Price')
plt.ylabel('Frequency')
plt.title('Car Prices')

plt.grid()
plt.show()

we should log the price values to work with it

In [ ]:
car_sales['log_price'] = np.log1p(car_sales['Price'])
car_sales['log_price']

In [ ]:
plt.figure(figsize=(10,6))

plt.hist(car_sales['log_price'], bins=20)
plt.xlabel('Log Price')
plt.ylabel('Frequency')
plt.title('Car Prices (log)')

plt.grid()
plt.show()

In [ ]:
#- Average price by manufacturer
avg_price_by_manufacturer = car_sales.groupby(['Manufacturer'])['Price'].mean().reset_index()
avg_price_by_manufacturer=avg_price_by_manufacturer.sort_values(by='Price',ascending=False)
avg_price_by_manufacturer

as we see the Manufacturer that has the heighest average price is `LAMBORGHINI`  
and we can see the gap between `LAMBORGHINI` and the secend `BENTLEY` it about 5 times

let's visualize the first and last 10 avg price 

In [ ]:
sample_avg_price = avg_price_by_manufacturer.head(15)
plt.figure(figsize=(20,12))

plt.plot(sample_avg_price['Manufacturer'], sample_avg_price['Price'])
plt.xlabel('Price')
plt.ylabel('Manufacturer')
plt.title('Avg car price for every Manufacturer (Heighest)')

plt.grid()
plt.show()

In [ ]:
sample_avg_price = avg_price_by_manufacturer.tail(15)
plt.figure(figsize=(20,12))

plt.plot(sample_avg_price['Manufacturer'], sample_avg_price['Price'])
plt.xlabel('Price')
plt.ylabel('Manufacturer')
plt.title('Avg car price for every Manufacturer (Lowest)')

plt.grid()
plt.show()

In [ ]:
plt.figure(figsize=(8,6))

plt.scatter(car_sales['Prod. year'],car_sales['log_price'])
plt.xlabel('Year of Production')
plt.ylabel('Sales')

plt.grid()
plt.show()

In [ ]:
plt.figure(figsize=(8,6))

sns.boxplot(x='Mileage',y='Price',data=car_sales)
plt.title('Box plot for Mileage vs Price')
plt.xticks(rotation=90)
plt.show()

In [ ]:
plt.figure(figsize=(10,6))

sns.countplot(x='Color',data=car_sales)
plt.title('Car Colors')
plt.xticks(rotation=90)
plt.show()